# Create snow-covered area (SCA) time series at Mores Creek Summit (WY24–25) using PlanetScope 4-band SR imagery

In [1]:
import os
from glob import glob
import rioxarray as rxr
import xarray as xr
import subprocess
import numpy as np
from tqdm import tqdm
import shutil
import matplotlib.pyplot as plt
import geopandas as gpd

# Inputs
base_dir = "/Users/rdcrlrka/Research/SIRO/MCS_SCA/"
raw_images_dir = os.path.join(base_dir, "raw_images")
aoi_file = os.path.join(base_dir, "MCS_outline", "basin_outline.shp")
cloud_masks_dir = os.path.join(base_dir, "PSS_cloud_masks")

# Outputs
mosaics_dir = os.path.join(base_dir, "PSS_image_mosaics")
mosaics_clip_dir = os.path.join(base_dir, "PSS_image_mosaics_clipped")
sca_dir = os.path.join(base_dir, "PSS_SCA_maps")


## Image pre-processing

### Compile raw images

In [ ]:
# --- If images are still in original folders, grab and compile ---
folders = sorted(glob(os.path.join(base_dir, "*udm2")))
os.makedirs(raw_images_dir, exist_ok=True)

if len(folders) > 0:
    # Iterate over zip folders
    for folder in tqdm(folders):
        # Move image files to out_folder
        image_files = [x for x in sorted(glob(os.path.join(folder, "PSScene", "*.tif"))) if "udm" not in os.path.basename(x)]
        for image_file in image_files:
            dest_file = os.path.join(raw_images_dir, os.path.basename(image_file))
            os.rename(image_file, dest_file)

        # Remove folder
        shutil.rmtree(folder)

        # Remove from the Trash too (lotta data)
        trash_folder = os.path.join(os.path.expanduser("~/.Trash"), os.path.basename(folder))
        if os.path.exists(trash_folder):
            shutil.rmtree(trash_folder)

        # Remove zip folder
        zip_folder = folder + ".zip"
        if os.path.exists(zip_folder):
            os.remove(zip_folder)
        trash_zip_folder = os.path.join(os.path.expanduser("~/.Trash"), os.path.basename(zip_folder))
        if os.path.exists(trash_zip_folder):
            os.remove(zip_folder)


In [ ]:
# --- Plot date coverage ---

# Locate raw images
pss_files = sorted(glob("raw_images/*.tif"))
print(f"Located {len(pss_files)} input images")

# Parse dates from the input files
all_dates = []
if pss_files:
    date_strings = [os.path.basename(x)[0:8] for x in pss_files]
    dates = [np.datetime64(f"{x[0:4]}-{x[4:6]}-{x[6:]}") for x in date_strings]
    all_dates += dates
    unique_dates = set(dates)
    print(f"Detected {len(unique_dates)} unique dates")
    unique_dates = sorted(np.array(list(unique_dates)))

    # Plot date coverage histogram
    plt.figure(figsize=(10,5))
    # make daily bins for the full date ranges
    bins = np.arange(min(all_dates), max(all_dates) + np.timedelta64(1, 'D'), np.timedelta64(1, 'D'))
    plt.hist(all_dates, bins=bins)
    plt.show()

### Create daily image mosaics

In [ ]:
os.makedirs(mosaics_dir, exist_ok=True)

# Iterate over unique dates
for unique_date in tqdm(unique_dates):
    # Get all images captured on date
    idate = np.argwhere(dates==unique_date).ravel()
    images_date = np.array(pss_files)[idate]

    # Check if mosaic already exists
    mosaic_file = os.path.join(mosaics_dir, f"{unique_date}_mosaic.tif")
    if os.path.exists(mosaic_file):
        print(f"Mosaic already exists for {unique_date}, skipping.")
        continue
    
    # Construct command
    print(f"Creating image mosaic for {unique_date}...")
    cmd = [
        'gdal_merge', 
        '-o', mosaic_file
        ] + list(images_date)

    # Run!
    subprocess.run(cmd, capture_output=False, check=True)
    


### Clip mosaics to AOI to save on space

In [ ]:
os.makedirs(mosaics_clip_dir, exist_ok=True)
mosaic_files = sorted(glob(os.path.join(mosaics_dir, '*.tif')))

# Iterate over mosaics
for mosaic_file in tqdm(mosaic_files, desc="Clipping Images"):
    # Define output file name
    base_name = os.path.splitext(os.path.basename(mosaic_file))[0]
    mosaic_clip_file = os.path.join(mosaics_clip_dir, f"{base_name}_clipped.tif")
    
    # Skip processing if output file already exists
    if os.path.exists(mosaic_clip_file):
        continue
        
    # Construct the gdalwarp command
    cmd = [
        "gdalwarp",
        "-cutline", aoi_file,
        "-crop_to_cutline", 
        "-dstnodata", "0", 
        "-co", "COMPRESS=DEFLATE",
        "-co", "TILED=YES", 
        mosaic_file,
        mosaic_clip_file
    ]
    
    try:
        # Run!
        subprocess.run(
            cmd, 
            check=True, 
            stdout=subprocess.PIPE, 
            stderr=subprocess.PIPE, 
            text=True
        )
        
    except subprocess.CalledProcessError as e:
        print(f"\n[ERROR] Failed to clip {os.path.basename(mosaic_file)}")
        print(f"Command Error Output:\n{e.stderr.strip()}")
        # Clean up partial outputs so subsequent runs retry it
        if os.path.exists(mosaic_clip_file):
            os.remove(mosaic_clip_file)
    


## Classify SCA

In [2]:
# Locate cloud masks
cloud_masks = sorted(glob(os.path.join(cloud_masks_dir, "*.gpkg")))
print(f"Located {len(cloud_masks)} cloud masks")

Located 22 cloud masks


In [ ]:
ndsi_threshold = -0.2

os.makedirs(sca_dir, exist_ok=True)
mosaic_clip_files = sorted(glob(os.path.join(mosaics_clip_dir, '*.tif')))

# Iterate over clipped image mosaics
for mosaic_clip_file in tqdm(mosaic_clip_files):
    date = os.path.basename(mosaic_clip_file)[0:10]

    # Check if SCA map already exists
    sca_file = os.path.join(sca_dir, f"{date}_SCA_NDSIthresh{ndsi_threshold}.tif")
    if os.path.exists(sca_file):
        continue

    with rxr.open_rasterio(mosaic_clip_file, masked=True, chunks="auto").squeeze() as mosaic_clip:     
        crs = mosaic_clip.rio.crs

        # Account for image scaler, make 0 values = NaN
        mosaic_clip = xr.where(mosaic_clip==1, np.nan, mosaic_clip / 1e4)

        # Check for cloud mask
        cloud_masks = [x for x in cloud_masks if date in os.path.basename(x)]
        if len(cloud_masks) > 0:
            print(f"Applying cloud mask for {date}")
            cloud_mask = gpd.read_file(cloud_masks[0])
            mosaic_clip = mosaic_clip.rio.clip(cloud_mask.geometry, cloud_mask.crs, drop=False)

        # Calculate modified NDSI
        g = mosaic_clip.isel(band=1)
        nir = mosaic_clip.isel(band=3)
        ndsi = (g-nir)/(g+nir)

        # Apply NDSI threshold
        sca = ndsi >= ndsi_threshold

        # reformat to integer with nodata value=255
        sca = xr.where(np.isnan(g), 255, sca).astype(np.uint8)
        sca = sca.rio.write_crs(crs)
        sca.rio.write_nodata(255)

        # Save to file
        sca.rio.to_raster(
            sca_file,
            crs=crs,
            nodata=255,
            dtype=np.uint8
        )
    

 57%|█████▋    | 61/107 [10:28<08:36, 11.23s/it]

: 